# 01 Quickstart: from the input data to the Eocene output

This notebook walks through the Eocene workflow of `ecefast` end to end:

1. get the input data,
2. write a configuration file from `configs/laptop.tmpl`,
3. list the pipeline steps,
4. check the inputs with a dry run,
5. run the whole pipeline,
6. look at the output tree and at `provenance.yml`.

It runs in about a minute. The per-component notebooks (`02_oifs.ipynb`, `03_nemo.ipynb`, `04_oasis_rnfm.ipynb`)
start from the configuration and the output written here.

**Environment.** Run it in the `eocene` conda environment with `ecefast` installed (`pip install -e .`, see
`docs/installation.rst`) and CDO on `PATH`.

**Paths** come from two environment variables, read in the next cell:

* `ECEFAST_TEST_DATA` (required): the directory holding the local data, as for `pytest -m local_data`:
  `ECE4-DATA-V3/`, `herold-et-al-2014/Supp/` and `data/` (the input of the NEMO domain pipeline and the Eocene
  domain built on Atos). In a full checkout this is the repository root.
* `ECEFAST_TUTORIAL_DIR` (optional): where the tutorials write their configuration and outputs. Default:
  `ecefast-tutorial` in the system temporary directory.

In [ ]:
import os
import tempfile
from pathlib import Path

import yaml

from ecefast.core.config import load_config
from ecefast.core.workspace import Context

DATA = Path(os.environ["ECEFAST_TEST_DATA"])  # ECE4-DATA-V3/, herold-et-al-2014/, data/
WORK = Path(os.environ.get("ECEFAST_TUTORIAL_DIR", Path(tempfile.gettempdir()) / "ecefast-tutorial"))
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = WORK / "laptop.yml"
REPO = Path.cwd().parents[1]  # this notebook lives in notebooks/tutorials/
print("data:", DATA)
print("work:", WORK)

## 1. Get the input data

The workflow reads three trees, all read-only:

| Directory | What | How to get it |
|---|---|---|
| `ECE4-DATA-V3/` | subset of the EC-Earth4 input tree (OIFS TL63L31, NEMO PALEORCA2, OASIS, runoff-mapper) | `scripts/get_ece4_data.sh` |
| `herold-et-al-2014/Supp/` | Herold et al. (2014) Eocene boundary conditions | from the HPC or the paper supplement |
| `data/domain/PALEORCA2/eocene/` | Eocene NEMO domain (`domain_cfg.nc`, `maskutil.nc`) built with DOMAINcfg on Atos | from the HPC |

`scripts/get_ece4_data.sh` copies the ECE4-DATA-V3 subset from a source given as an argument or in `ECE4_DATA_SRC`
(run it from the repository root, not from this notebook):

```bash
ECE4_DATA_SRC=user@host:/path/to/ECE4-DATA-V3 ./scripts/get_ece4_data.sh
```

Check that everything is there:

In [ ]:
needed = [
    DATA / "ECE4-DATA-V3",
    DATA / "herold-et-al-2014" / "Supp",
    DATA / "data" / "domain" / "PALEORCA2" / "eocene",
    DATA / "data" / "PALEORCA2" / "coords_ori.nc",
]
for path in needed:
    print("ok     " if path.exists() else "MISSING", path.relative_to(DATA))

## 2. Write a configuration file

On the command line you would copy the template and edit the paths by hand:

```bash
cp configs/laptop.tmpl configs/laptop.yml   # configs/*.yml is ignored by git
```

Here the same is done in Python: read the template, set the paths, write `laptop.yml` into the working directory.
(Writing the YAML back drops the template comments; every key is described in `docs/configuration.rst`.)

The sections follow the time periods:

* `machine`: where the tools run, and `tmpdir`, where every command keeps its intermediate files.
* `grids`: the model resolution, shared by every period; `grids.nemo.coords_ori` is read to build a domain.
* `present`: the present-day period. `input` is the EC-Earth input tree (read-only), `output` the tree it makes;
  `present.nemo` builds the present-day NEMO domain (`ecefast nemo run`), used in `03_nemo.ipynb`.
* `eocene`: the Eocene period (`ecefast eocene run`). It starts from `input` (the present-day tree,
  `"{present.input}"`), reads the Herold data from `herold` and writes `output`; `nemo.domain` is the Eocene NEMO
  domain built on Atos, which the `domain` step installs.

Here the outputs go to `OUTPUT/present` and `OUTPUT/eocene` in the working directory, the intermediate files to
`tmp/`.

In [ ]:
config = yaml.safe_load((REPO / "configs" / "laptop.tmpl").read_text())

config["machine"]["tmpdir"] = str(WORK / "tmp")
config["grids"]["nemo"]["coords_ori"] = str(DATA / "data" / "PALEORCA2" / "coords_ori.nc")
config["present"]["input"] = str(DATA / "ECE4-DATA-V3")
config["present"]["output"] = str(WORK / "OUTPUT" / "present")
config["eocene"]["herold"] = str(DATA / "herold-et-al-2014" / "Supp")
config["eocene"]["output"] = str(WORK / "OUTPUT" / "eocene")
config["eocene"]["nemo"]["domain"] = str(DATA / "data" / "domain" / "PALEORCA2" / "eocene")

CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG.read_text()[:600], "...")

`load_config` validates the file against the schema: unknown keys, wrong types and missing input directories are
errors. Keys left out take their defaults:

In [ ]:
cfg = load_config(str(CONFIG))
print(cfg.machine)
print(cfg.grids)
print(cfg.present)
print(cfg.eocene)

## 3. The pipeline steps

`ecefast eocene steps` lists the steps in the order they run: their component, what each one needs (steps that
must run before), reads and writes: path templates with placeholders of the configuration. `{input}`, `{herold}`,
`{output}` are keys of the running period (here `eocene`), dotted ones such as `{grids.nemo.name}` keys of the file.

* `stage` copies the base files from the input tree to the output tree;
* `domain` installs the Eocene NEMO domain and checks its vertical grid;
* `oasis`, `oifs`, `nemo`, `rnfm` rewrite the files of one component each.

In [ ]:
!ecefast eocene steps

## 4. Dry run

`--dry-run` checks every input of the selected steps and shows the plan, without writing anything. All missing
inputs are reported at once, before any step runs.

In [ ]:
!ecefast eocene run -c {CONFIG} --dry-run

## 5. Full run

`--overwrite` lets `stage` replace an output directory that is not empty (here: a previous run of this notebook).
The run takes about a minute.

In [ ]:
!ecefast eocene run -c {CONFIG} --overwrite

## 6. The output

The output tree mirrors the input tree, so EC-Earth finds every file where it expects it. Files that a step
rewrites keep their name and relative path; variants of a file add a suffix (`woa13-levitus-L31_woa-pi.nc`).
What each step wrote:

In [ ]:
from ecefast.periods.eocene.pipeline import PIPELINE

ctx = Context(cfg, "eocene")
OUTPUT = Path(cfg.eocene.output)
assert (OUTPUT / "provenance.yml").is_file(), "the run did not finish: see the log above"

for step in PIPELINE:
    print(step.name)
    for template in step.writes:
        path = Path(ctx.expand(template))
        size = "directory" if path.is_dir() else f"{path.stat().st_size / 1e6:.1f} MB"
        print(f"    {path.relative_to(OUTPUT)}  ({size})")

Every run writes `provenance.yml` into the output directory: the steps that ran, the configuration file and its
checksum, the `ecefast` version and git commit, the Python, CDO and ecCodes versions, start and end time.

In [ ]:
print((OUTPUT / "provenance.yml").read_text())

## 7. The same from Python

`ecefast eocene run` calls `ecefast.periods.eocene.pipeline.run`. `select` is the `--steps` option: rerun some
steps on an existing output (here as a dry run, so nothing changes).

In [ ]:
from ecefast.core import setup_logger
from ecefast.periods.eocene.pipeline import run

setup_logger(level="INFO")
run(str(CONFIG), select=["oifs", "nemo"], dry_run=True)

## Next

* `02_oifs.ipynb`: land-sea mask, orography, vegetation and albedo of the atmosphere;
* `03_nemo.ipynb`: the Eocene ocean domain, the vertical grid check and the NEMO forcing files;
* `04_oasis_rnfm.ipynb`: the coupler restart and the river routing.